# 05 — Portfolio Risk Dashboard

## Why this project exists

Portfolio managers care about more than return. They need to understand risk concentration, drawdowns, correlations and volatility.

This notebook builds a small multi-asset portfolio dashboard using Yahoo Finance ETF data. The goal is to demonstrate investment analytics that would be useful for portfolio support or fund analysis.

In [ ]:
!pip install yfinance plotly -q

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.express as px

pd.options.display.float_format = "{:,.4f}".format

## 1. Define a sample portfolio

I use a simple example portfolio instead of personal holdings. This makes the project safe to publish and easy to reproduce.

In [ ]:
weights = pd.Series({
    "SPY": 0.35,
    "QQQ": 0.20,
    "TLT": 0.15,
    "GLD": 0.15,
    "EFA": 0.10,
    "UUP": 0.05,
})

prices = yf.download(list(weights.index), start="2018-01-01", auto_adjust=True, progress=False)["Close"].dropna()
returns = prices.pct_change().dropna()
portfolio_returns = returns.mul(weights, axis=1).sum(axis=1)
weights

## 2. Calculate core portfolio metrics

I focus on metrics that are intuitive for investment teams:

- annualized return
- annualized volatility
- Sharpe ratio
- max drawdown
- current drawdown
- contribution to volatility proxy

In [ ]:
equity = (1 + portfolio_returns).cumprod()
drawdown = equity / equity.cummax() - 1

ann_return = portfolio_returns.mean() * 252
ann_vol = portfolio_returns.std() * np.sqrt(252)
sharpe = ann_return / ann_vol
max_dd = drawdown.min()
current_dd = drawdown.iloc[-1]

metrics = pd.Series({
    "annual_return": ann_return,
    "annual_volatility": ann_vol,
    "sharpe_ratio": sharpe,
    "max_drawdown": max_dd,
    "current_drawdown": current_dd,
})
metrics

## 3. Visualize equity curve and drawdown

Drawdown matters because it shows the psychological and risk-management pain of a portfolio.

In [ ]:
px.line(equity, title="Sample portfolio equity curve").show()
px.area(drawdown, title="Sample portfolio drawdown").show()

## 4. Correlation and risk concentration

A portfolio can look diversified by number of holdings but still be concentrated in risk. Correlation helps identify hidden concentration.

In [ ]:
corr = returns.corr()
px.imshow(corr, text_auto=True, title="Asset correlation matrix").show()

asset_vol = returns.std() * np.sqrt(252)
risk_table = pd.DataFrame({
    "weight": weights,
    "annual_volatility": asset_vol,
    "simple_weight_x_vol_proxy": weights * asset_vol,
}).sort_values("simple_weight_x_vol_proxy", ascending=False)
risk_table

## 5. My conclusion

This dashboard provides a practical overview of portfolio health:

- allocation
- realized performance
- drawdown
- volatility
- correlation
- simple risk concentration

## Next iterations

- Add factor exposures
- Add rolling beta versus SPY
- Add monthly performance attribution
- Add stress-test scenarios
- Add Streamlit interface